# Laboratorio de IA ética: modelo base del Titanic

Predecir la supervivencia de pasajeros con un bosque aleatorio. Ejecuta las celdas en orden.
El preprocesamiento aprende únicamente del conjunto de entrenamiento; la evaluación utiliza validación.
El conjunto de prueba queda reservado para una evaluación final posterior.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/memo124/Laboratorio_IA_etica/blob/main/main.ipynb)

## 1. Dependencias

Este notebook utiliza las librerías incluidas en Google Colab.
Todos los imports se concentran en la siguiente celda.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Configuración

Rutas, columnas y parámetros compartidos del experimento.

In [ ]:
DATA_PATH = Path("train.csv")
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
RANDOM_STATE = 42
TARGET_COLUMN = "Survived"
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked"]
TEST_FRACTION = 0.20
VALIDATION_FRACTION = 0.20
N_ESTIMATORS = 100

## 3. Carga de datos

Se utiliza el CSV local. Si no existe, se descarga y se guarda para próximas ejecuciones.

In [ ]:
def load_dataset(path: Path, url: str) -> pd.DataFrame:
    """Lee los datos locales o descarga y guarda una copia del CSV."""
    if path.exists():
        return pd.read_csv(path)

    dataset = pd.read_csv(url)
    dataset.to_csv(path, index=False)
    return dataset


data = load_dataset(DATA_PATH, DATA_URL)

## 4. Exploración inicial

Vista previa, tipos de datos y valores faltantes por columna.

In [ ]:
display(data.head())
data.info()
display(data.isna().sum().rename("Valores faltantes").to_frame())

## 5. Separación de entradas y etiqueta

`Survived` es la etiqueta: 1 indica supervivencia y 0 indica fallecimiento. Se conservan todas las demás columnas, incluyendo `Ticket`, para compartir las mismas particiones entre integrantes. El preprocesador seleccionará únicamente las siete variables del Baseline.

In [ ]:
features = data.drop(columns=[TARGET_COLUMN])
target = data[TARGET_COLUMN]

## 6. División de datos

Entrenamiento (60 %), validación (20 %) y prueba (20 %). La estratificación conserva aproximadamente la proporción de supervivientes en cada conjunto.

In [ ]:
X_remaining, X_test, y_remaining, y_test = train_test_split(
    features,
    target,
    test_size=TEST_FRACTION,
    stratify=target,
    random_state=RANDOM_STATE,
)

# La validación representa el 25 % del 80 % restante: un 20 % del total.
X_train, X_val, y_train, y_val = train_test_split(
    X_remaining,
    y_remaining,
    test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
    stratify=y_remaining,
    random_state=RANDOM_STATE,
)

split_sizes = pd.Series(
    {"Entrenamiento": len(X_train), "Validación": len(X_val), "Prueba": len(X_test)},
    name="Filas",
)
split_summary = split_sizes.to_frame()
split_summary["Porcentaje"] = (100 * split_sizes / len(data)).round(1)
display(split_summary)

# Verificamos que las particiones compartidas conserven filas y etiquetas alineadas.
partitions = [(X_train, y_train), (X_val, y_val), (X_test, y_test)]
for partition_features, partition_target in partitions:
    assert partition_features.index.equals(partition_target.index)
    assert "Ticket" in partition_features.columns
    assert TARGET_COLUMN not in partition_features.columns

partition_indices = [set(partition_features.index) for partition_features, _ in partitions]
assert partition_indices[0].isdisjoint(partition_indices[1])
assert partition_indices[0].isdisjoint(partition_indices[2])
assert partition_indices[1].isdisjoint(partition_indices[2])
assert set.union(*partition_indices) == set(data.index)
assert sum(len(partition_features) for partition_features, _ in partitions) == len(data)


## 7. Preprocesamiento

Las variables numéricas se imputan con la mediana; las categóricas, con la moda, y luego se codifican con one-hot. `Pclass` se trata como categoría. Esta celda solo define las transformaciones.

In [ ]:
numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, NUMERIC_COLUMNS),
        ("categorical", categorical_transformer, CATEGORICAL_COLUMNS),
    ],
    verbose_feature_names_out=False,
)

## 8. Construcción del modelo

Un único pipeline aplica el preprocesamiento y el clasificador, evitando repetir transformaciones manualmente al predecir.

In [ ]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=N_ESTIMATORS,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

## 9. Entrenamiento

El ajuste del preprocesamiento y del clasificador utiliza únicamente los datos de entrenamiento.

In [ ]:
baseline_model.fit(X_train, y_train)

feature_names = baseline_model.named_steps["preprocessor"].get_feature_names_out()
print(f"Variables resultantes: {len(feature_names)}")
print(", ".join(feature_names))

## 10. Preparación de validación y prueba

Aplicamos las reglas aprendidas durante el entrenamiento con `transform`, sin volver a ajustarlas. Las matrices preparadas conservan el orden de las filas de sus respectivas particiones. Transformar prueba no implica evaluarla: sus etiquetas no se utilizan aquí.


In [ ]:
fitted_preprocessor = baseline_model.named_steps["preprocessor"]
X_val_prepared = fitted_preprocessor.transform(X_val)
X_test_prepared = fitted_preprocessor.transform(X_test)

# Ambas matrices deben tener las mismas variables que recibió el clasificador.
expected_features = baseline_model.named_steps["classifier"].n_features_in_
for prepared, original in [(X_val_prepared, X_val), (X_test_prepared, X_test)]:
    assert prepared.shape == (len(original), expected_features)
    assert not pd.isna(prepared).any()

print(f"Validación preparada: {X_val_prepared.shape}")
print(f"Prueba preparada: {X_test_prepared.shape}")


## 11. Evaluación en validación

El F1 y el reporte de clasificación establecen la referencia del modelo base. El conjunto de prueba ya está preparado, pero no se utiliza para evaluar el modelo ni tomar decisiones en esta etapa.

In [ ]:
y_val_pred = baseline_model.predict(X_val)
validation_f1 = f1_score(y_val, y_val_pred)

print(f"F1 en validación: {validation_f1:.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_val, y_val_pred))

# Integrante 2: nuevas variables a partir del precio y la edad

Usaremos las mismas particiones del Baseline. Los gráficos, las medianas y los límites
se obtienen únicamente de entrenamiento. Después aplicamos esas mismas reglas a validación y prueba.
Estas transformaciones no demuestran por sí solas que el modelo mejore; eso se comparará más adelante.

## 12. Distribución del precio del boleto


Un histograma agrupa valores en intervalos y cuenta cuántos pasajeros hay en cada uno.
`bins=30` divide los precios en 30 intervalos. Omitimos los datos faltantes solo para dibujar;
no eliminamos pasajeros de las particiones.

Al ejecutar, observa si las barras se concentran en precios bajos y se extienden hacia precios altos.
Esa forma se llama cola hacia la derecha. Probaremos un logaritmo para comprimir los precios altos.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(X_train["Fare"].dropna(), bins=30, edgecolor="black")
plt.title("Precio del boleto en entrenamiento")
plt.xlabel("Precio del boleto (Fare)")
plt.ylabel("Cantidad de pasajeros")
plt.tight_layout()
plt.show()

## 13. Copias y valores faltantes


`.copy()` permite añadir columnas sin modificar los datos del Baseline.
La mediana es el valor central de una lista ordenada. La calculamos solo con entrenamiento,
y usamos ese mismo valor para completar los faltantes en las tres particiones.
Las series auxiliares contienen los valores completados; las columnas originales siguen intactas.

In [ ]:
X_train_engineered = X_train.copy()
X_val_engineered = X_val.copy()
X_test_engineered = X_test.copy()

fare_median = X_train["Fare"].median()
age_median = X_train["Age"].median()

In [ ]:
fare_train_filled = X_train["Fare"].fillna(fare_median)
fare_val_filled = X_val["Fare"].fillna(fare_median)
fare_test_filled = X_test["Fare"].fillna(fare_median)

age_train_filled = X_train["Age"].fillna(age_median)
age_val_filled = X_val["Age"].fillna(age_median)
age_test_filled = X_test["Age"].fillna(age_median)

## 14. Transformación logarítmica


`np.log1p` calcula el logaritmo natural de uno más el valor. Comprime los precios altos
y admite un precio de cero, porque `log1p(0)` es cero.
Guardamos el resultado en `Fare_log`, conservando `Fare` para futuras comparaciones.

In [ ]:
X_train_engineered["Fare_log"] = np.log1p(fare_train_filled)
X_val_engineered["Fare_log"] = np.log1p(fare_val_filled)
X_test_engineered["Fare_log"] = np.log1p(fare_test_filled)

## 15. Histogramas antes y después


Comparamos únicamente entrenamiento, usando `Fare` ya imputado como punto de partida.
Los ejes horizontales tienen unidades distintas: precio a la izquierda y logaritmo a la derecha.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].hist(fare_train_filled, bins=30, edgecolor="black")
axes[0].set_title("Antes: precio del boleto")
axes[0].set_xlabel("Fare imputado")
axes[0].set_ylabel("Cantidad de pasajeros")

axes[1].hist(X_train_engineered["Fare_log"], bins=30, edgecolor="black")
axes[1].set_title("Después: logaritmo del precio")
axes[1].set_xlabel("Fare_log")
axes[1].set_ylabel("Cantidad de pasajeros")

plt.tight_layout()
plt.show()

**Cómo interpretar la comparación:** el logaritmo acerca los precios muy altos al resto de los valores.
Al ejecutar, compara la concentración de barras y la cola derecha para describir si la distribución queda más equilibrada.
Que cambie la forma del histograma no garantiza una mejora en las predicciones.

## 16. Revisión de valores extremos


Un boxplot resume una distribución: la línea central marca la mediana y la caja contiene
la mitad central de los datos. Los puntos más alejados ayudan a detectar posibles valores extremos.
Un precio alto o una edad poco frecuente no necesariamente son errores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].boxplot(X_train["Fare"].dropna())
axes[0].set_title("Precios en entrenamiento")
axes[0].set_xticks([1], ["Fare"])
axes[0].set_ylabel("Precio del boleto")

axes[1].boxplot(X_train["Age"].dropna())
axes[1].set_title("Edades en entrenamiento")
axes[1].set_xticks([1], ["Age"])
axes[1].set_ylabel("Edad (años)")

plt.tight_layout()
plt.show()

## 17. Límites para recortar extremos


El percentil 1 es un valor por debajo del cual queda aproximadamente el 1 % de los datos;
el percentil 99 deja aproximadamente el 99 % por debajo.
`quantile(0.01)` y `quantile(0.99)` calculan esos límites ignorando los faltantes.
Los calculamos sobre las columnas originales de entrenamiento y no los recalculamos con validación o prueba.

In [ ]:
fare_lower = X_train["Fare"].quantile(0.01)
fare_upper = X_train["Fare"].quantile(0.99)
age_lower = X_train["Age"].quantile(0.01)
age_upper = X_train["Age"].quantile(0.99)

winsor_limits = pd.DataFrame({
    "Variable": ["Fare", "Age"],
    "Percentil 1": [fare_lower, age_lower],
    "Percentil 99": [fare_upper, age_upper],
})
display(winsor_limits)

## 18. Winsorización del precio y la edad


Winsorizar significa recortar valores extremos sin eliminar filas.
`.clip(lower=..., upper=...)` reemplaza los valores menores al límite inferior por ese límite,
y hace lo mismo con los mayores al superior. Los valores intermedios no cambian.
Aplicamos los límites de entrenamiento a las series ya imputadas y creamos columnas nuevas.

In [ ]:
X_train_engineered["Fare_winsor"] = fare_train_filled.clip(lower=fare_lower, upper=fare_upper)
X_val_engineered["Fare_winsor"] = fare_val_filled.clip(lower=fare_lower, upper=fare_upper)
X_test_engineered["Fare_winsor"] = fare_test_filled.clip(lower=fare_lower, upper=fare_upper)

In [ ]:
X_train_engineered["Age_winsor"] = age_train_filled.clip(lower=age_lower, upper=age_upper)
X_val_engineered["Age_winsor"] = age_val_filled.clip(lower=age_lower, upper=age_upper)
X_test_engineered["Age_winsor"] = age_test_filled.clip(lower=age_lower, upper=age_upper)

## 19. Boxplots antes y después


Cada fila compara una variable de entrenamiento antes y después del recorte.
`sharey="row"` mantiene la misma escala vertical en cada comparación.
Usamos los valores imputados en los gráficos de antes para observar únicamente el efecto del recorte.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey="row")

axes[0, 0].boxplot(fare_train_filled)
axes[0, 0].set_title("Antes: precio imputado")
axes[0, 0].set_xticks([1], ["Fare"])
axes[0, 0].set_ylabel("Precio del boleto")

axes[0, 1].boxplot(X_train_engineered["Fare_winsor"])
axes[0, 1].set_title("Después: precio recortado")
axes[0, 1].set_xticks([1], ["Fare_winsor"])

axes[1, 0].boxplot(age_train_filled)
axes[1, 0].set_title("Antes: edad imputada")
axes[1, 0].set_xticks([1], ["Age"])
axes[1, 0].set_ylabel("Edad (años)")

axes[1, 1].boxplot(X_train_engineered["Age_winsor"])
axes[1, 1].set_title("Después: edad recortada")
axes[1, 1].set_xticks([1], ["Age_winsor"])

plt.tight_layout()
plt.show()

**Cómo interpretar la comparación:** después del recorte, los valores quedan dentro de los percentiles guardados, sin perder pasajeros.
Aún pueden aparecer puntos fuera de los bigotes: el boxplot utiliza un criterio diferente de los percentiles 1 y 99.
Al ejecutar, observa cuánto cambia cada variable; no es necesario que desaparezcan todos los puntos extremos.

Las copias `X_train_engineered`, `X_val_engineered` y `X_test_engineered` conservan las columnas originales,
incluyendo `Ticket`, y añaden `Fare_log`, `Fare_winsor` y `Age_winsor`.
El siguiente integrante podrá usarlas con las etiquetas `y_train`, `y_val` y `y_test`, respectivamente.